# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# `metadata` is a DatasetMetadata object, convert to dict for pretty print
metadata_dict = dataset.metadata.to_json()

print(f"Dataset name: {metadata_dict['name']}")
print(f"Description: {metadata_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs (referenced by `@id`).

In [ ]:
# List all available record sets by `@id`
record_sets = dataset.metadata.record_sets
if record_sets == []:
    print("No record sets explicitly defined; loading from available schema artifacts.")

# Get full metadata dump for inspection
metadata_json = dataset.metadata.to_json_ld()

# Collect all RecordSet entities by their @id
def collect_recordset_ids(metadata_json):
    recordset_ids = []
    if isinstance(metadata_json, dict):
        if metadata_json.get("@type") in ["RecordSet", "cr:RecordSet"]:
            recordset_ids.append(metadata_json["@id"])
        for v in metadata_json.values():
            recordset_ids.extend(collect_recordset_ids(v))
    elif isinstance(metadata_json, list):
        for v in metadata_json:
            recordset_ids.extend(collect_recordset_ids(v))
    return recordset_ids

recordset_ids = collect_recordset_ids(metadata_json)
recordset_ids = list(set(recordset_ids))  # unique

if not recordset_ids:
    print("No record sets with '@id'. Using dataset.records() directly to list possible columns.")
    # Try retrieving data to inspect columns
    try:
        preview = list(dataset.records())[:2]
        print('Preview of data fields:')
        for k in preview[0].keys():
            print('-', k)
    except Exception as e:
        print('No data preview available:', e)
else:
    print('RecordSet @ids found:')
    for rsid in recordset_ids:
        print(' -', rsid)

# For each RecordSet, print fields and columns (referenced by @id)
def find_record_set_entities(metadata_json, target_id):
    if isinstance(metadata_json, dict):
        if metadata_json.get("@id") == target_id:
            return metadata_json
        for v in metadata_json.values():
            result = find_record_set_entities(v, target_id)
            if result:
                return result
    elif isinstance(metadata_json, list):
        for v in metadata_json:
            result = find_record_set_entities(v, target_id)
            if result:
                return result

# Print fields for each record set
for rsid in recordset_ids:
    rec_set = find_record_set_entities(metadata_json, rsid)
    print(f'\nRecordSet {rsid}:')
    fields = rec_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    field_ids = []
    for f in fields:
        if isinstance(f, dict) and '@id' in f:
            field_ids.append(f['@id'])
        elif isinstance(f, str):
            field_ids.append(f)
    print('  Fields:')
    for fid in field_ids:
        print('   -', fid)
    # Try also extracting columns if present
    columns = rec_set.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print('  Columns:')
        for col in columns:
            if isinstance(col, dict) and '@id' in col:
                print('   -', col['@id'])
            elif isinstance(col, str):
                print('   -', col)

# If no defined record sets, just note the fact
if not recordset_ids:
    print("No record sets with `@id` found. Use dataset.records() for extraction.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, no record sets are defined with '@id'. Thus, use dataset.records() directly.
try:
    preview_records = list(dataset.records())
    df = pd.DataFrame(preview_records)
    print(f"Columns in the data: {df.columns.tolist()}")
    df.head()
except Exception as e:
    print(f"Could not load records: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select a likely numeric field for analysis
print("Columns for selection:", list(df.columns))

# Let's attempt to select one among common proteomics fields
# You might need to adjust the name depending on the output in the above cell
possible_numeric_fields = [
    'cr:field:coverage_percent',
    'cr:field:mw',             # Molecular weight
    'cr:field:peptide_count'   # Peptide count
]
numeric_field = None
for fld in possible_numeric_fields:
    if fld in df.columns:
        numeric_field = fld
        break
# If not found, try fallback to first numeric-like column
if not numeric_field:
    for fld in df.columns:
        if df[fld].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[fld]):
            numeric_field = fld
            break

if not numeric_field:
    raise ValueError('No numeric field found for analysis.')
else:
    print(f'Using numeric field: {numeric_field}')

# Remove null or non-numeric values if any
df_clean_num = pd.to_numeric(df[numeric_field], errors='coerce')
df_valid = df[df_clean_num.notnull()].copy()
df_valid[numeric_field] = df_clean_num[df_clean_num.notnull()]

threshold = df_valid[numeric_field].mean()  # Example: mean used as a threshold
filtered_df = df_valid[df_valid[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f} (mean):")
print(filtered_df[[numeric_field]].head())

# Normalization (standard score)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

# Group by another likely attribute (e.g. sample, experiment, or accession)
possible_group_fields = [
    'cr:field:accession',
    'cr:field:sample',
    'cr:field:protein_id'
]
group_field = None
for gf in possible_group_fields:
    if gf in df.columns:
        group_field = gf
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print("No available group field found; skipping group-by example.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df_valid[numeric_field], bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If group_field is available, boxplot
if group_field:
    # Show only the top 10 most frequent groups for clarity
    top_groups = df_valid[group_field].value_counts().head(10).index
    plt.figure(figsize=(12,6))
    sns.boxplot(x=group_field, y=numeric_field, data=df_valid[df_valid[group_field].isin(top_groups)])
    plt.title(f'{numeric_field} by top {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.
- The dataset includes records with fields detailing protein abundance, modifications, and other key attributes from mass spectrometry data.
- Data can be filtered, normalized, and visualized using standard Python tools.

Further analyses are possible depending on your research questions—refer to the Croissant schema and field `@id`s to ensure reproducibility when programmatically referencing dataset elements.